<div style='text-align:center;padding:30px;background:linear-gradient(135deg,#101014 0%,#191920 100%);border:2px solid #ff684a;border-radius:16px;margin:10px 0;box-shadow:0 10px 30px rgba(255,104,74,.22);font-family:sans-serif;'>
  <div style='color:#ff684a;font-size:.78em;font-weight:800;letter-spacing:.16em;text-transform:uppercase;'>SUPERIMAGEM</div>
  <h1 style='color:white;margin:8px 0;font-size:2.5em;'>Ampliação profissional com GPU T4</h1>
  <p style='color:#ffb4a4;margin:0;font-size:1.08em;'>2×, 4× e 8× • transparência PNG preservada • By Diego Gomes</p>
</div>

> Confirme que aparece **T4** no botão **Conectar** e execute somente a célula abaixo.

In [ ]:
#@title ▶ INICIAR SUPERIMAGEM GPU T4
REPO_URL = 'https://github.com/diegogomespi/superimagem-colab.git'
BRANCH = 'main'
PROJECT_DIR = '/content/SuperImagem'
PORT = 7860

import os, subprocess, sys, time
import torch
from IPython.display import HTML, clear_output, display

assert torch.cuda.is_available(), 'GPU não encontrada. Em Ambiente de execução, selecione a GPU T4 e tente novamente.'
gpu = torch.cuda.get_device_properties(0)
print(f'✓ GPU pronta: {gpu.name} | {gpu.total_memory / 1024**3:.1f} GB de VRAM')

print('1/3 Instalando o motor de ampliação...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'spandrel==0.4.1', 'python-multipart==0.0.20'])

print('2/3 Atualizando o SuperImagem pelo GitHub...')
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    subprocess.check_call(['git', 'fetch', '--depth', '1', 'origin', BRANCH], cwd=PROJECT_DIR)
    subprocess.check_call(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=PROJECT_DIR)
else:
    import shutil
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, PROJECT_DIR])

print('3/3 Abrindo a interface...')
if 'server' in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)
os.chdir(PROJECT_DIR)
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'colab.app:app', '--host', '0.0.0.0', '--port', str(PORT)])

import requests
for _ in range(90):
    try:
        health = requests.get(f'http://127.0.0.1:{PORT}/api/health', timeout=2).json()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('O servidor não iniciou. Consulte as últimas mensagens acima.')

from google.colab import output
page_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
clear_output(wait=True)
display(HTML(f"""
<div style='background:radial-gradient(circle at top,#26120d 0%,#0d0d12 60%);padding:40px;border-radius:18px;text-align:center;font-family:Inter,sans-serif;max-width:720px;margin:0 auto;border:1px solid #4d2118;box-shadow:0 24px 60px rgba(0,0,0,.4)'>
  <div style='width:18px;height:18px;border-radius:50%;background:#39d98a;box-shadow:0 0 20px #39d98a;margin:0 auto 14px'></div>
  <h2 style='color:white;margin:0 0 8px;font-size:2em'>SuperImagem está online!</h2>
  <p style='color:#ffb4a4;margin:0 0 24px'>GPU conectada: {health['gpu']['name']}</p>
  <a href='{page_url}' target='_blank' style='display:inline-block;background:linear-gradient(90deg,#ff684a,#f23d68);color:white;padding:16px 34px;border-radius:9px;text-decoration:none;font-weight:800;font-size:16px'>ABRIR SUPERIMAGEM EM NOVA ABA →</a>
  <p style='color:#817c86;font-size:12px;margin:24px 0 0'>Mantenha esta sessão do Colab aberta enquanto estiver usando.</p>
</div>
"""))
